In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from keras.models import Sequential
from keras.layers import GRU, Dense, Dropout
from keras.optimizers import Adam
from keras.callbacks import EarlyStopping
from sklearn.model_selection import TimeSeriesSplit

def smape(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    return 100 * np.mean(
        2 * np.abs(y_pred - y_true) /
        (np.abs(y_true) + np.abs(y_pred))
    )

def mase(y_true, y_pred, y_train):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    y_train = np.array(y_train)

    naive_error = np.mean(np.abs(np.diff(y_train)))
    return np.mean(np.abs(y_true - y_pred)) / naive_error

def theil_u(y_true, y_pred):
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    denominator = np.sqrt(np.mean(y_true ** 2) + np.mean(y_pred ** 2))
    u = rmse / denominator
    return u

# Load dataset
df = pd.read_excel('output_week.xlsx')

# Count records and sort by week
data = df['WEEK'].value_counts().reset_index()
data.columns = ['WEEK', 'DATA']

# Convert week identifier to a date representing the beginning of the week
data['WEEK_period'] = pd.to_datetime(
    data['WEEK'] + '-1',
    format='%G-%V-%u',
    errors='coerce'
)
data = data.sort_values('WEEK_period').reset_index(drop=True)

# ------------------------------
# Smoothing (moving average)
# ------------------------------
data['DATA_smooth'] = data['DATA'].rolling(
    window=5,
    center=True
).mean()

data['DATA_smooth'].fillna(method='bfill', inplace=True)
data['DATA_smooth'].fillna(method='ffill', inplace=True)

# Data normalization
scaler = MinMaxScaler(feature_range=(0.1, 1))
data['y'] = scaler.fit_transform(data[['DATA_smooth']])

# ------------------------------
# Create lagged features
# ------------------------------
n_lags = 11  # Alternative values such as 12, 16, or 26 may also be tested

for i in range(1, n_lags + 1):
    data[f'lag_{i}'] = data['y'].shift(i)

data = data.dropna().reset_index(drop=True)

# ------------------------------
# Prepare X and y
# ------------------------------
lag_cols = [f'lag_{i}' for i in range(n_lags, 0, -1)]

X = data[lag_cols].values.astype(np.float32)
y = data['y'].values.astype(np.float32)

# Reshape for GRU input
X = X.reshape((X.shape[0], X.shape[1], 1))

/var/folders/36/lb3mdhtj2lldbybnz877q_hr0000gn/T/ipykernel_1673/110638030.py:57: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data['DATA_smooth'].fillna(method='bfill', inplace=True)
/var/folders/36/lb3mdhtj2lldbybnz877q_hr0000gn/T/ipykernel_1673/110638030.py:57: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  data['DATA_smooth'].fillna(method='bfill', inplace=True)
/var/folders/36/lb3mdhtj2lldbybnz877q_hr0000gn/T/ipykernel_1673/110638030.p

In [2]:
tscv = TimeSeriesSplit(n_splits=10)


rmse_gru = []
mae_gru = []
mase_gru = []
smape_gru = []


for fold, (train_idx, test_idx) in enumerate(tscv.split(X), 1):

    print(f"\nFold {fold}")

    # ------------------------------
    # Split temporal
    # ------------------------------

    X_train = X[train_idx]
    y_train = y[train_idx]

    X_test = X[test_idx]
    y_test = y[test_idx]


    # ------------------------------
    # Create GRU Model
    # ------------------------------

    model = Sequential()


    # First GRU layer

    model.add(
        GRU(
            units=256,
            activation='tanh',
            return_sequences=True,
            input_shape=(
                X_train.shape[1],
                X_train.shape[2]
            )
        )
    )


    model.add(
        Dropout(0.1)
    )


    # Second GRU layer

    model.add(
        GRU(
            units=256,
            activation='tanh'
        )
    )


    model.add(
        Dropout(0.1)
    )


    # Output layer

    model.add(
        Dense(
            1,
            activation='linear'
        )
    )


    # ------------------------------
    # Compile
    # ------------------------------

    model.compile(
        optimizer=Adam(
            learning_rate=0.001
        ),
        loss='mse'
    )


    # ------------------------------
    # Early stopping
    # ------------------------------

    early_stop = EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True
    )


    # ------------------------------
    # Training
    # ------------------------------

    history = model.fit(
        X_train,
        y_train,
        epochs=100,
        batch_size=16,
        validation_split=0.2,
        callbacks=[early_stop],
        verbose=0
    )


    # ------------------------------
    # Forecasting
    # ------------------------------

    y_pred = model.predict(
        X_test,
        verbose=0
    ).flatten()



    # ------------------------------
    # Return to original scale
    # ------------------------------

    y_test_real = scaler.inverse_transform(
        y_test.reshape(-1,1)
    ).flatten()


    y_pred_real = scaler.inverse_transform(
        y_pred.reshape(-1,1)
    ).flatten()


    y_train_real = scaler.inverse_transform(
        y_train.reshape(-1,1)
    ).flatten()



    # ------------------------------
    # Metrics
    # ------------------------------

    mase_value = mase(
        y_test_real,
        y_pred_real,
        y_train_real
    )


    rmse_value = np.sqrt(
        mean_squared_error(
            y_test_real,
            y_pred_real
        )
    )


    mae_value = mean_absolute_error(
        y_test_real,
        y_pred_real
    )


    smape_value = smape(
        y_test_real,
        y_pred_real
    )


    # ------------------------------
    # Save metrics
    # ------------------------------

    rmse_gru.append(rmse_value)
    mae_gru.append(mae_value)
    mase_gru.append(mase_value)
    smape_gru.append(smape_value)


    print(
        f"RMSE={rmse_value:.4f} | "
        f"MAE={mae_value:.4f} | "
        f"MASE={mase_value:.4f}"
    )



# ------------------------------
# Save GRU results
# ------------------------------

results_gru = pd.DataFrame({

    "Fold": range(1, len(rmse_gru)+1),
    "RMSE": rmse_gru,
    "MAE": mae_gru,
    "MASE": mase_gru,
    "SMAPE": smape_gru

})


results_gru.to_csv(
    "GRU_results.csv",
    index=False
)


Fold 1


/Users/octaciliofsilvajunior/miniforge3/lib/python3.10/site-packages/keras/src/layers/rnn/rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


RMSE=4.9117 | MAE=3.9115 | MASE=2.6638

Fold 2


/Users/octaciliofsilvajunior/miniforge3/lib/python3.10/site-packages/keras/src/layers/rnn/rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


RMSE=2.4819 | MAE=1.9626 | MASE=1.3803

Fold 3


...

RMSE=0.8756 | MAE=0.7068 | MASE=0.6818

Fold 9


/Users/octaciliofsilvajunior/miniforge3/lib/python3.10/site-packages/keras/src/layers/rnn/rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


RMSE=1.1071 | MAE=0.9061 | MASE=0.9033

Fold 10


/Users/octaciliofsilvajunior/miniforge3/lib/python3.10/site-packages/keras/src/layers/rnn/rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


RMSE=0.8152 | MAE=0.6803 | MASE=0.6865
